# Contrastes de H3 y H5 (revisión del 1-oct-2026)

Responde los comentarios 12, 14 y 15 del profesor guía.

| Sección | Hipótesis | Método |
|---|---|---|
| 1 | H5: el traspaso a SLEP no se asocia a una recuperación distinta | Emparejamiento por GSE y puntaje previo |
| 2 | H3: la recuperación es más lenta a menor GSE | Modelo de medición con el nivel previo en la ecuación de la pendiente |
| 3 | H3: el gradiente es mayor en Matemática que en Lectura | Diferencia de los gradientes entre modelos |
| 4 | Comentario 12 | Contribución SHAP del GSE con bootstrap, como análisis complementario |

**Requiere** los notebooks 02 y 06 ejecutados y `pip install xgboost`. Las secciones 2 y 3 ajustan dos modelos bayesianos de unos 7 minutos cada uno.

> **Revisión del 24-oct-2026 (auditoría de cuadernos y código).** Dos cambios:

1. **MDE de H5.** Ahora se calcula con el error estándar bootstrap del propio ATT emparejado (2,8 × EE). La fórmula anterior suponía dos muestras independientes de 115 y 712 escuelas, pero el ATT compara 113 pares. Salida esperada: ATT y su intervalo no cambian (0,048 [−0,473; 0,604]); la tabla trae la columna nueva `EE_bootstrap` (≈ 0,27) y el MDE de la recuperación pasa de 0,64 a **≈ 0,77**; el de la caída, de 4,53 a ≈ 3,9.
2. **Sensibilidad de H3 al GSE previo a la pandemia** (sección 3b, nueva; unos 8 minutos). Salida esperada: Matemática, Alto − Bajo = **−1,37 [−2,55; −0,17]**, P(Alto > Bajo) = 0,02: con el GSE previo, el GSE Bajo se recupera *más rápido*. Lectura: +1,17 [0,25; 2,08]. Diferencia Matemática − Lectura: −2,54 [−3,99; −1,07]. H3 sigue sin sostenerse, y con más fuerza; lo que cambia es que en Matemática el resultado deja de ser "sin diferencia".

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")
B, K = sp.bayes, sp.contrastes

In [2]:
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))
res_m = sp.io_datos.leer_csv(cfg.ruta_salida("resultados_mate_2m.csv"))
res_l = sp.io_datos.leer_csv(cfg.ruta_salida("resultados_lect_2m.csv"))
esc = sp.io_datos.leer_csv(cfg.ruta_salida("bayes_escuelas_mate.csv"))
largo = sp.io_datos.leer_csv(cfg.ruta_salida(sp.config.NOMBRES_SALIDA["rbd"]))
largo, _ = sp.estandarizacion.estandarizar_canonico(largo, variables=["cod_depe1"])
print("Escuelas con estimación bayesiana:", len(esc))

Escuelas con estimación bayesiana: 2861


## 1. H5 — Traspaso a SLEP (comentario 14)

Tratados: establecimientos con dependencia modal municipal en 2015–2018 y SLEP en 2022–2025. Controles: municipales en ambos períodos. Cada tratado se empareja con el control de su mismo GSE más cercano en puntaje previo (con reemplazo, caliper de 0,25 desviaciones estándar). El resultado es la estimación del modelo de medición del notebook 06.

**Diferencia mínima detectable (MDE).** Es 2,8 veces el error estándar bootstrap del ATT: la prueba tiene un 80 % de potencia, al 5 % bilateral, para detectar diferencias de este tamaño. (Hasta el 24-oct-2026 se usaba la fórmula de dos muestras independientes, que subestimaba el MDE.) Una diferencia estimada menor que la MDE y con intervalo que cruza el cero es compatible con H5, pero solo descarta efectos mayores que la MDE.

In [3]:
cl = K.clasificar_traspasos(largo)
print(cl["grupo_h5"].value_counts().to_string())
d5 = (esc.merge(cl, on="rbd").merge(ancho[["rbd", "cod_grupo_cat"]], on="rbd")
         .merge(res_m[["rbd", "puntaje_pre"]], on="rbd"))
pd.DataFrame([K.h5_emparejado(d5, y) for y in ("recuperacion_bayes", "caida_bayes")])

grupo_h5
otro        2041
control      712
traspaso     115


,resultado,n_traspaso,n_control,n_emparejados,ATT,IC95,SMD_pre_antes,SMD_pre_despues,EE_bootstrap,MDE_80pct
0,recuperacion_bayes,115,712,113,0.048,"[-0.473, 0.604]",-0.238,0.001,0.275,0.769
1,caida_bayes,115,712,113,1.173,"[-1.561, 3.869]",-0.238,0.001,1.382,3.869


**Lectura.** Si el emparejamiento lleva la diferencia estandarizada del puntaje previo cerca de cero y el intervalo del efecto cruza el cero con una MDE pequeña respecto de la recuperación media (unos 3,5 puntos/año), H5 queda respaldada: el traspaso no se asocia a una recuperación distinta. Es una asociación, no un efecto causal: los traspasos no fueron aleatorios.

**Esta ejecución (auditoría).** MDE ≈ 0,77 puntos por año, cerca de un quinto de la recuperación media: se descartan efectos del traspaso mayores que unos 0,8 puntos por año, no los menores. H5 se contrastó solo en Matemática; en Lectura, el cuaderno 11 muestra una asociación creíble de los SLEP con una recuperación más lenta (−0,92 [−1,62; −0,23]), que conviene contrastar aquí con el mismo emparejamiento.

## 2. H3 — Gradiente socioeconómico a igual nivel previo

**Por qué esta especificación.** Regresionar las medias a posteriori del notebook 06 sobre el GSE produce un artefacto: el modelo ya contrae cada pendiente hacia la media de su GSE, de modo que la regresión recupera la estructura del propio modelo, con intervalos artificialmente estrechos. La prueba correcta incluye el nivel previo dentro de la ecuación de la pendiente; así, la pendiente de cada GSE se estima a igual puntaje de partida.

Se ajusta para ambas asignaturas con 1.000 + 1.000 iteraciones por cadena.

In [4]:
def panel(materia):
    anios = [2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025]
    return pd.concat([pd.DataFrame({
        "rbd": ancho["rbd"], "cod_grupo_cat": ancho["cod_grupo_cat"], "anio": t,
        "y": pd.to_numeric(ancho[f"prom_{materia}2m_rbd_{t}"], errors="coerce"),
        "n": pd.to_numeric(ancho[f"nalu_{materia}2m_rbd_{t}"], errors="coerce")})
        for t in anios]).dropna(subset=["y"])

ajustes = {}
for materia in ("mate", "lect"):
    datos = B.preparar_datos(panel(materia), col_n="n")
    ajustes[materia] = B.ajustar(datos, tendencia_pre=True, controlar_pre=True,
                                 num_warmup=1000, num_samples=1000)
    print(materia, ajustes[materia].diagnosticos)

15:30:13 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


15:30:13 | INFO    | simce_panel.bayes | Datos del modelo: 2861 escuelas, 22606 observaciones.


15:30:15 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.


15:33:19 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 184.3, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 628}


15:33:19 | INFO    | simce_panel.bayes | 6 escuelas excluidas por no tener puntaje en 2022.


mate {'segundos': 184.3, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 628}


15:33:19 | INFO    | simce_panel.bayes | Datos del modelo: 2860 escuelas, 22595 observaciones.


15:36:00 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 160.6, 'divergencias': 0, 'r_hat_max': 1.004, 'n_eff_min': 466}


lect {'segundos': 160.6, 'divergencias': 0, 'r_hat_max': 1.004, 'n_eff_min': 466}


In [5]:
def gradiente(mcmc):
    tb = np.asarray(mcmc.get_samples()["th_b"]) * 50
    q = lambda v: [round(float(np.percentile(v, p)), 2) for p in (3, 50, 97)]
    tabla = pd.DataFrame({g: q(tb[:, i]) for i, g in enumerate(B.ORDEN_GSE)},
                         index=["IC 3 %", "mediana", "IC 97 %"]).T
    return tabla, tb[:, 4] - tb[:, 0]

difs = {}
for materia, mcmc in ajustes.items():
    tabla, difs[materia] = gradiente(mcmc)
    print(f"=== {materia}: pendiente de recuperación por GSE a igual nivel previo (pts/año)")
    display(tabla)
    v = difs[materia]
    print(f"Alto − Bajo: {np.median(v):+.2f} [{np.percentile(v, 3):+.2f}; "
          f"{np.percentile(v, 97):+.2f}] | P(Alto > Bajo) = {(v > 0).mean():.3f}\n")

=== mate: pendiente de recuperación por GSE a igual nivel previo (pts/año)


,IC 3 %,mediana,IC 97 %
Bajo,2.94,3.55,4.14
Medio bajo,3.15,3.57,4.01
Medio,3.08,3.53,4.01
Medio alto,2.56,3.24,3.93
Alto,2.83,3.64,4.43


Alto − Bajo: +0.09 [-1.07; +1.23] | P(Alto > Bajo) = 0.556

=== lect: pendiente de recuperación por GSE a igual nivel previo (pts/año)


,IC 3 %,mediana,IC 97 %
Bajo,1.02,1.50,1.97
Medio bajo,1.80,2.17,2.51
Medio,1.74,2.14,2.53
Medio alto,1.85,2.42,3.00
Alto,2.41,3.03,3.63


Alto − Bajo: +1.53 [+0.64; +2.40] | P(Alto > Bajo) = 1.000



## 3. H3 — ¿Es mayor el gradiente en Matemática que en Lectura?

Los dos modelos se ajustan por separado, de modo que sus posteriores son independientes y la diferencia se obtiene combinando muestras.

In [6]:
rng = np.random.default_rng(0)
n = min(len(difs["mate"]), len(difs["lect"]))
dif = rng.permutation(difs["mate"])[:n] - rng.permutation(difs["lect"])[:n]
print(f"Gradiente Matemática − Lectura: {np.median(dif):+.2f} "
      f"[{np.percentile(dif, 3):+.2f}; {np.percentile(dif, 97):+.2f}]")
print(f"P(gradiente de Matemática mayor) = {(dif > 0).mean():.3f}")

Gradiente Matemática − Lectura: -1.42 [-2.82; -0.00]
P(gradiente de Matemática mayor) = 0.029


## 3b. Sensibilidad de H3 al GSE previo a la pandemia (auditoría del 24-oct-2026)

Las secciones anteriores usan el GSE del año más reciente, posterior a 2022. Un 25 % de las escuelas tiene un GSE modal 2015–2018 distinto. Aquí se repite el contraste con el GSE previo (las escuelas sin GSE previo conservan el más reciente). Pon `SENSIBILIDAD_GSE_PREVIO = False` para omitirlo.

In [7]:
SENSIBILIDAD_GSE_PREVIO = True
if SENSIBILIDAD_GSE_PREVIO:
    if "cod_grupo_cat" not in largo.columns:
        largo, _ = sp.estandarizacion.estandarizar_canonico(largo, variables=["cod_grupo"])
    gse_pre = B.gse_modal_previo(largo)

    def panel_pre(materia):
        L = panel(materia)
        return L.assign(cod_grupo_cat=L["rbd"].map(gse_pre).fillna(L["cod_grupo_cat"]))

    difs_pre = {}
    for materia in ("mate", "lect"):
        datos_p = B.preparar_datos(panel_pre(materia), col_n="n")
        m_p = B.ajustar(datos_p, tendencia_pre=True, controlar_pre=True,
                        num_warmup=1000, num_samples=1000)
        print(materia, m_p.diagnosticos)
        tabla_p, difs_pre[materia] = gradiente(m_p)
        v = difs_pre[materia]
        print(f"=== {materia} con GSE previo: Alto − Bajo {np.median(v):+.2f} "
              f"[{np.percentile(v, 3):+.2f}; {np.percentile(v, 97):+.2f}] | "
              f"P(Alto > Bajo) = {(v > 0).mean():.3f}")
        display(tabla_p)
    rng = np.random.default_rng(0)
    n = min(len(difs_pre["mate"]), len(difs_pre["lect"]))
    dif = rng.permutation(difs_pre["mate"])[:n] - rng.permutation(difs_pre["lect"])[:n]
    print(f"Gradiente Matemática − Lectura con GSE previo: {np.median(dif):+.2f} "
          f"[{np.percentile(dif, 3):+.2f}; {np.percentile(dif, 97):+.2f}]")

15:36:01 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


15:36:01 | INFO    | simce_panel.bayes | Datos del modelo: 2861 escuelas, 22606 observaciones.


15:39:08 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 186.3, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 647}


mate {'segundos': 186.3, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 647}
=== mate con GSE previo: Alto − Bajo -1.37 [-2.58; -0.16] | P(Alto > Bajo) = 0.019


,IC 3 %,mediana,IC 97 %
Bajo,3.73,4.32,4.89
Medio bajo,3.36,3.80,4.25
Medio,2.59,3.07,3.54
Medio alto,2.12,2.79,3.52
Alto,2.08,2.95,3.80


...

15:39:09 | INFO    | simce_panel.bayes | Datos del modelo: 2860 escuelas, 22595 observaciones.


15:42:33 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 204.3, 'divergencias': 0, 'r_hat_max': 1.001, 'n_eff_min': 493}


lect {'segundos': 204.3, 'divergencias': 0, 'r_hat_max': 1.001, 'n_eff_min': 493}
=== lect con GSE previo: Alto − Bajo +1.19 [+0.26; +2.05] | P(Alto > Bajo) = 0.991


,IC 3 %,mediana,IC 97 %
Bajo,1.34,1.81,2.28
Medio bajo,1.93,2.30,2.66
Medio,1.39,1.78,2.18
Medio alto,1.98,2.55,3.14
Alto,2.33,3.01,3.65


Gradiente Matemática − Lectura con GSE previo: -2.54 [-4.03; -1.07]


**Lectura.** Salida esperada: Matemática −1,37 [−2,55; −0,17]; Lectura +1,17 [0,25; 2,08]. En Matemática, el GSE Bajo previo incluye unas 180 escuelas que después pasaron a Medio bajo; venían mejorando y se recuperan rápido. Parte de esa recuperación puede reflejar un cambio en la composición de sus estudiantes y no solo aprendizaje, de modo que el signo negativo no se lee como "las escuelas pobres se recuperan mejor". La conclusión robusta es la otra: **a igual nivel previo, el GSE alto no se recupera más rápido en Matemática con ninguna de las dos clasificaciones.**

**Lectura.** H3 tiene dos partes: (a) la recuperación es más lenta a menor GSE y (b) ese gradiente es mayor en Matemática. Con los datos de esta tesis, a igual nivel previo, (a) se observa en Lectura pero no en Matemática, y (b) no se sostiene: la evidencia apunta en la dirección contraria. Es un resultado legítimo: H3 se formuló antes de ver los datos y se reporta tal como resultó.

## 4. Contribución SHAP con bootstrap (comentario 12)

Análisis complementario. Se ajusta un modelo de árboles potenciados por asignatura y se mide el gradiente SHAP del GSE (contribución media en GSE Alto menos la de GSE Bajo). Los mismos establecimientos se remuestrean en ambas asignaturas en cada réplica, de modo que el intervalo de la diferencia es válido.

**Advertencia.** Antes de interpretar SHAP debe verificarse que el modelo predice fuera de muestra. Con un R² por comunas cercano o inferior a cero, las atribuciones describen ruido. Además, el GSE y el puntaje previo están muy correlacionados (r ≈ 0,76), y SHAP reparte el crédito entre ellos de forma inestable. Por eso la prueba principal de H3 es la de las secciones 2 y 3.

In [8]:
num = ["puntaje_pre", "matricula_pre", "densidad_5km", "dist_referencia_km",
       "dist_centroide_comuna_km"]
conj = {}
for materia, r in (("mate", res_m), ("lect", res_l)):
    r = r[r["muestra_analitica"].astype(str).str.lower().isin(["true", "1"])]
    da = sp.dataset.ensamblar_dataset(r, ancho, materia=materia).set_index("rbd")
    X = K.preparar_matriz(da, num)
    conj[materia] = (X, da["recuperacion"])
    print(materia, "R² fuera de muestra (validación por comunas):",
          K.r2_por_bloques(X, da["recuperacion"], da["comuna_dir"]))

15:42:35 | INFO    | simce_panel.dataset | Variables geográficas calculadas para 3158 establecimientos.


15:42:35 | INFO    | simce_panel.dataset | Dataset analítico: 2821 filas x 39 columnas.


mate R² fuera de muestra (validación por comunas): -0.038


15:42:38 | INFO    | simce_panel.dataset | Variables geográficas calculadas para 3158 establecimientos.


15:42:38 | INFO    | simce_panel.dataset | Dataset analítico: 2818 filas x 35 columnas.


lect R² fuera de muestra (validación por comunas): -0.054


In [9]:
shap_res = K.gradiente_shap_bootstrap(conj, n_boot=500)
shap_res

{'n_establecimientos': 2818,
 'n_boot': 500,
 'mate': {'gradiente': 0.089, 'IC95': [-0.693, 0.92]},
 'lect': {'gradiente': 1.259, 'IC95': [0.291, 2.059]},
 'diferencia_mate_menos_lect': {'estimacion': -1.17,
  'IC95': [-1.891, -0.06],
  'prob_mayor_que_0': 0.018}}

## 5. Síntesis

| Hipótesis | Resultado esperado del contraste |
|---|---|
| H5 | Diferencia emparejada con intervalo que cruza el cero; MDE ≈ 0,77 con el error estándar bootstrap: compatible con H5 en Matemática |
| H3 (a) | Gradiente a igual nivel previo: presente en Lectura, ausente en Matemática (con GSE previo, invertido en Matemática) |
| H3 (b) | Gradiente de Matemática no mayor que el de Lectura: no se sostiene |
| Comentario 12 | SHAP coincide en dirección, pero con R² negativo no constituye evidencia por sí solo |

Verifica estas conclusiones contra las salidas de tu ejecución antes de trasladarlas al documento.